# Delivering Outputs to Object Storage

## What you'll learn

- Send a Modal-executed command op's outputs to an allowed S3-compatible
  bucket prefix with `output_store`
- Follow the delivery: the worker uploads a tarball under your prefix
  and the client fetches it through a presigned URL
- Verify and clean up the delivered objects

**Prerequisites:** [Running on Modal](04-modal-execution.ipynb).
**Estimated time:** 10 minutes.
**GPU required:** No


:::{note}
This tutorial requires Modal credentials, a deployed `wait_tool`
endpoint whose deployment carries an object-store secret (setup below),
and an S3-compatible bucket. The walkthrough uses Cloudflare R2; AWS S3
and MinIO work identically. Code cells are shown for reference and are
not executed in the docs build.
:::


## Why a store

By default a Modal-executed op returns its outputs inline, bounded at
100 MB. Naming an `output_store` removes the bound: the worker delivers
the output tarball straight to your bucket and only a pointer rides the
result. The destination is **request data**, but it must remain under
an output prefix baked into the endpoint's `data_policy`. Two callers
may choose different descendants of an allowed prefix; widening access
requires a redeploy. The
full contract (credentials, lifecycle, external consumers) is in
[Object-store output delivery](../../how-to-guides/configuring-execution.md#object-store-output-delivery).


## One-time setup

The worker uploads with its **own** credentials, injected as a Modal
Secret. Create the secret once, name it in the op's config, and deploy:

```bash
modal secret create r2-artisan \
  AWS_ACCESS_KEY_ID=... AWS_SECRET_ACCESS_KEY=... \
  AWS_REGION=auto AWS_ENDPOINT_URL=https://<account>.r2.cloudflarestorage.com
```

```python
class MyTool(OperationDefinition):
    ...
    compute_provider = ComputeProvider(
        modal=ModalComputeConfig(
            secrets=["r2-artisan"],
            data_policy=ToolEndpointDataPolicy(
                output_allowlist=(
                    "s3://my-bucket/artisan-tutorial",
                    "https://<account>.r2.cloudflarestorage.com",
                ),
            ),
        )
    )
```

```bash
artisan modal deploy my_tool
```

For the `wait_tool` example endpoint used here, the repo's live
integration test deploys it with the secret and this tutorial's output
prefix attached:
`pixi run -e dev test-modal-endpoint`.

This notebook also reads your bucket coordinates from the environment
(or the repo-root `.env`):

```bash
# .env
AWS_ACCESS_KEY_ID=...            # used below to verify the delivery
AWS_SECRET_ACCESS_KEY=...
ARTISAN_S3_ENDPOINT_URL=https://<account>.r2.cloudflarestorage.com
ARTISAN_S3_BUCKET=my-bucket
```


In [ ]:
from __future__ import annotations

from artisan.operations.examples import DataGenerator, WaitTool
from artisan.orchestration import PipelineManager, StepDisposition
from artisan.schemas import (
    ComputeProvider,
    ModalComputeConfig,
    ToolEndpointDataPolicy,
)
from artisan.utils import env_or_dotenv, tutorial_setup
from artisan.visualization import inspect_step

env = tutorial_setup("modal_r2_outputs", clean=True)

In [ ]:
import os
import uuid

BUCKET = env_or_dotenv("ARTISAN_S3_BUCKET")
ENDPOINT = env_or_dotenv("ARTISAN_S3_ENDPOINT_URL")
assert BUCKET, "set ARTISAN_S3_BUCKET (see setup above)"
assert ENDPOINT, "set ARTISAN_S3_ENDPOINT_URL (see setup above)"

# ambient credentials for the verification cells at the end — the same
# variable shapes the worker's Modal Secret injects on the other side
for key in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
    os.environ.setdefault(key, env_or_dotenv(key) or "")
os.environ.setdefault("AWS_ENDPOINT_URL", ENDPOINT)

OUTPUT_STORE = f"s3://{BUCKET}/artisan-tutorial/{uuid.uuid4().hex[:8]}"
OUTPUT_POLICY = ToolEndpointDataPolicy(
    output_allowlist=(f"s3://{BUCKET}/artisan-tutorial", ENDPOINT),
)
print(f"outputs will be delivered under {OUTPUT_STORE}")

## Run on Modal, deliver to the bucket

`output_store` joins the modal config like any other field. Here it
rides a per-step override within the policy already baked at deploy
time. The matching client-side policy catches mistakes early; the
worker's baked copy remains authoritative. A step without the override
still returns outputs inline:


`skip_cache=True` forces a new remote execution and delivery when you rerun
the example.

In [ ]:
pipeline = PipelineManager.create(
    name="modal_r2_outputs",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

gen = pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 2, "seed": 42},
)

wait = pipeline.run(
    operation=WaitTool,
    skip_cache=True,
    name="wait",
    inputs={"dataset": gen.output("datasets")},
    params={"seconds": 1},
    compute_provider=ComputeProvider(
        active="modal",
        modal=ModalComputeConfig(output_store=OUTPUT_STORE, data_policy=OUTPUT_POLICY),
    ),
)

result = pipeline.finalize()
assert result["overall_success"], result
assert wait.disposition is StepDisposition.EXECUTED
for completed_step in (gen, wait):
    artifacts = inspect_step(
        env.delta_root,
        completed_step.step_number,
        pipeline_run_id=pipeline.config.pipeline_run_id,
    )
    assert artifacts.height == 2, (completed_step.step_name, artifacts)
print("pipeline complete")

## What happened

For each artifact, the worker ran the tool, tarred its outputs, and
uploaded `<output_store>/<op-name>/<uuid>.tar.gz` with the deployment's
secret. The result carried no bulk bytes — only the tarball's URI plus
a presigned download URL (valid 7 days, matching how long Modal retains
the result). The pipeline client fetched through that URL and committed
artifacts exactly as an inline run would — the lifecycle downstream is
identical.

The same pointer serves consumers outside artisan: anyone polling the
endpoint gets the presigned URL and fetches results with one plain HTTP
GET, no storage credentials. Artisan's data client rejects redirects.
A caller can send a presigned PUT URL only when its exact origin is in
the baked output policy — see
[capability mode](../../how-to-guides/configuring-execution.md#presigned-puts-and-external-consumers-capability-mode).


## Verify the delivery

Two artifacts fanned out as two endpoint calls, so two tarballs landed
under the prefix. List them, and build a link to browse them in your
provider's web console:


In [ ]:
from urllib.parse import urlparse

import s3fs

fs = s3fs.S3FileSystem()
delivered = fs.find(OUTPUT_STORE.removeprefix("s3://"))
assert len(delivered) == 2, delivered
assert all(key.endswith(".tar.gz") for key in delivered)
for key in delivered:
    print(key)

host = urlparse(ENDPOINT).netloc
prefix = OUTPUT_STORE.removeprefix(f"s3://{BUCKET}/")
if host.endswith(".r2.cloudflarestorage.com"):
    account_id = host.split(".", 1)[0]
    console = f"https://dash.cloudflare.com/{account_id}/r2/default/buckets/{BUCKET}"
elif host.endswith(".amazonaws.com"):
    console = f"https://s3.console.aws.amazon.com/s3/buckets/{BUCKET}?prefix={prefix}/"
else:
    console = None  # self-hosted stores (MinIO, ...) serve their own console
print(
    "\nbrowse the delivered files:", console or f"open your store's console for {host}"
)

## Clean up

Artisan never deletes delivered tarballs — pair real destination
prefixes with a bucket lifecycle policy. For the tutorial prefix, remove
the objects directly:


In [ ]:
output_prefix = OUTPUT_STORE.removeprefix("s3://")
if fs.exists(output_prefix):
    fs.rm(output_prefix, recursive=True)
    print("removed", OUTPUT_STORE)

## Summary

- `output_store` on the modal config delivers a Modal-executed op's
  outputs beneath a prefix allowed by its baked `data_policy`, bypassing
  the 100 MB inline bound while retaining stored-archive safety budgets
- The worker uploads with its Modal Secret's credentials; consumers
  fetch through an allowed presigned-URL origin, credential-free
- Changing the allowed prefixes or origins requires redeploying; callers
  cannot widen policy through request fields
- Without `output_store`, behavior is unchanged: outputs return inline,
  bounded at 100 MB

## Next steps

- [Object-store output delivery](../../how-to-guides/configuring-execution.md#object-store-output-delivery)
  — the full contract: endpoint policy, IAM scoping, lifecycle, external consumers
- [Configure S3-Compatible Storage](../../how-to-guides/configuring-s3.md)
  — putting the pipeline's own storage (Delta tables, staging, files)
  on a bucket
- [Running on Modal](04-modal-execution.ipynb) — the endpoint model
  this builds on
- [Execution Flow](../../concepts/execution-flow.md) — how compute
  routing and output delivery fit the execution model